<a href="https://colab.research.google.com/github/ananthakrishnanm010/Helmet-and-Rider-Safety-Violation-Detection/blob/main/HelmetSafety_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing Ultralytics package
-Ultralytics is a Python package, which gives you tools to train and use YOLO models.

In [ ]:
!pip install ultralytics

In [ ]:
!pip install -q -U kagglehub


# Dataset Reading

In [ ]:

import os
import yaml
import kagglehub

# Download the public Kaggle dataset
dataset_root = kagglehub.dataset_download(
    "febinnemmanuel/helmet-safety-yolov8-dataset"
)

print("Downloaded to:", dataset_root)

# Inspect the downloaded directory
print("Contents:", os.listdir(dataset_root))

# Find data.yaml anywhere inside the downloaded dataset
DATASET_PATH = None

for root, dirs, files in os.walk(dataset_root):
    if "data.yaml" in files:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "data.yaml was not found inside the downloaded dataset."
    )

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

print("DATASET_PATH:", DATASET_PATH)
print("YAML_PATH:", YAML_PATH)
print("Dataset contents:", os.listdir(DATASET_PATH))


In [ ]:
import os
import shutil

SOURCE_PATH = "/kaggle/input/helmet-safety-yolov8-dataset/content/helmet_yolo"
DATASET_PATH = "/content/helmet_yolo"

if not os.path.exists(DATASET_PATH):
    shutil.copytree(SOURCE_PATH, DATASET_PATH)

print("Dataset copied to:", DATASET_PATH)
print("Contents:", os.listdir(DATASET_PATH))

In [ ]:
import yaml
import os

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

data["path"] = DATASET_PATH

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(open(YAML_PATH).read())

In [ ]:
for split in ["train", "val", "test"]:
    folder = os.path.join(DATASET_PATH, "images", split)
    print(split, "exists:", os.path.isdir(folder))

    if os.path.isdir(folder):
        print("Number of files:", len(os.listdir(folder)))

In [ ]:
import os
import yaml

print("Dataset path:", DATASET_PATH)
print("\nDataset YAML:")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

print(data)

for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATASET_PATH, "images", split)
    label_dir = os.path.join(DATASET_PATH, "labels", split)

    print(f"\n{split.upper()}")
    print("Images:", len(os.listdir(image_dir)) if os.path.isdir(image_dir) else "MISSING")
    print("Labels:", len(os.listdir(label_dir)) if os.path.isdir(label_dir) else "MISSING")

# Yolov8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

# Model Baseline Training

In [ ]:
results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640
)

In [ ]:
metrics = model.val()
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

### Model Evaluation on the Test Set

In [ ]:
# Load the best weights from our baseline training run
best_model_path = '/content/runs/detect/train-2/weights/best.pt'
best_model = YOLO(best_model_path)

# Evaluate on the test split by setting split='test'
test_metrics = best_model.val(split='test')

print("\n--- Test Dataset Performance Metrics ---")
print(f"mAP@50: {test_metrics.box.map50:.4f}")
print(f"mAP@50-95: {test_metrics.box.map:.4f}")
print(f"Precision (mp): {test_metrics.box.mp:.4f}")
print(f"Recall (mr): {test_metrics.box.mr:.4f}")

In [ ]:
# ==========================================
# WEEK 2: FINE-TUNING WITH DATA AUGMENTATION
# ==========================================
from ultralytics import YOLO

# Initialize from pre-trained or baseline weights
augmented_model = YOLO("yolov8n.pt")

augmented_results = augmented_model.train(
    data=YAML_PATH,
    epochs=60,
    imgsz=640,
    batch=16,
    # Data Augmentation parameters (Week 2 Requirement)
    mosaic=1.0,        # Combines 4 images into one (crucial for small distant pillions)
    mixup=0.15,        # Blends frames to learn through occlusions
    degrees=10.0,      # Rotation jitter for angled road cameras
    scale=0.5,         # Scale jitter (handling distant vs close-up riders)
    fliplr=0.5,        # Horizontal flip for bidirectional traffic
    hsv_h=0.015,       # Hue shift
    hsv_s=0.7,         # Saturation shift
    hsv_v=0.4,         # Brightness shift (simulating Indian sunlight & shadows)
    patience=15
)

# Evaluate the augmented model on the test split
augmented_test_metrics = augmented_model.val(split='test')

print("\n--- Baseline vs. Fine-Tuned (Augmented) Comparison ---")
print(f"Baseline Test mAP@50:  0.8759  --> Fine-Tuned: {augmented_test_metrics.box.map50:.4f}")
print(f"Baseline Test Recall:  0.7910  --> Fine-Tuned: {augmented_test_metrics.box.mr:.4f}")
print(f"Baseline Without-Helmet Recall: 0.6940")
